# OctoSense × FiftyOne — Multimodal Robotics Data Curation

*Companion notebook to the blog post "Eight Sensors, One Dataset, Zero Blind Spots." Run it top to
bottom to reproduce the full demo: a browsable, searchable, QA-able FiftyOne dataset built from real
OctoSense sensor data. Everything is downloaded live from the public dataset — no credentials, no
local files required.*

This notebook turns one or more sequences from **[OctoSense](https://huggingface.co/datasets/anthonytec2/OctoSense)**
— a time-synchronized car / boat / quadruped sensor dataset — into a fully explorable
**FiftyOne** dataset, then launches the App as a dataset-level *curation and label-QA* surface.

OctoSense already ships excellent per-sequence tooling (a Rerun timeline viewer, a caption
semantic-search CLI). What it *doesn't* ship is a place to explore the whole collection at once:
filter by conditions, QA the ground-truth labels frame by frame, cluster in embedding space,
and search by natural language — all cross-linked. That's the gap this notebook fills.

**What you get in the App**

- Each sample is one **rectified left-RGB frame** (sampled per LiDAR scan) — the frame the dataset
  renders all its ground truth into, so everything aligns pixel-for-pixel.
- **Semantic segmentation** (Cityscapes-19) as a `Segmentation` mask — for label QA.
- **LiDAR depth** as a `Heatmap` — spot holes, blooming, dynamic-object gaps.
- **Ego-motion optical flow** magnitude as a second `Heatmap` (derived at ingest).
- **Scene captions** (Gemma VLM) as sample text.
- **Two similarity indexes**: a CLIP index (native App text search + an embedding UMAP plot),
  and OctoSense's own **precomputed Qwen3 caption embeddings** loaded as a second index for
  caption-space retrieval.
- **Rich metadata** per frame (speed, turn rate, day/night) and per sequence
  (distance, GPS bbox, sensor dropout) for filtering, plus a **map** panel from GPS.

**Design note — two search modes, on purpose.** OctoSense's caption embeddings are 4096-d Qwen3
*text* embeddings; they live in a text-only space and can't be probed with CLIP image-text prompts.
So we build a CLIP index over the pixels (for App-native prompt search and the embedding plot)
*and* register the Qwen vectors as their own index (for "find frames whose caption is like this
one"). Different spaces, complementary questions.

**Re-run friendly.** Every expensive step is idempotent: downloaded files, ingested frames, and
brain runs are all cached and skipped on a second run. You can safely re-execute the whole notebook
after a kernel restart without repeating the slow parts. Set `FORCE_REBUILD = True` (Section 1) to
wipe and start clean.

---
> **Runs on:** macOS or Linux, Python 3.10–3.12, `fiftyone==1.18`. CPU-only is fine (the first run
> pulls a small CLIP model from the FiftyOne zoo). Setup instructions are in Section 0 — you only
> need a working Python; the notebook creates its own environment recipe.
>
> **~15–20 min end to end** on a laptop: a few minutes to download two OctoSense sequences (cached
> after that), a few minutes to ingest ~400 frames, and a couple to build the search indexes.


## 0 · Environment setup

The cleanest way to run this is a **dedicated virtual environment** so nothing conflicts with other
projects (in particular, the OctoSense reader needs `torch==2.12` via `torchcodec`, which can clash
with other torch versions if you share an env).

**One-time setup** — run this in a terminal, then launch Jupyter from the activated env and select
the `octosense-demo` kernel:

```bash
python3 -m venv .venv-octosense
source .venv-octosense/bin/activate            # Windows: .venv-octosense\Scripts\activate
pip install "fiftyone==1.18.0"
pip install "torch==2.12.0" "torchvision==0.27.0" "torchcodec==0.14.0" \
    --index-url https://download.pytorch.org/whl/cpu
pip install h5py hdf5plugin opencv-python-headless pyproj umap-learn \
    huggingface_hub ipykernel ipywidgets
python -m ipykernel install --user --name octosense-demo --display-name "octosense-demo"
```

The cell below verifies you're on a sane environment and installs anything missing, so you can also
just run the notebook and let it self-check.

In [ ]:
import sys
print("Interpreter:", sys.executable)
# Soft check: warn (don't hard-fail) if this doesn't look like a dedicated venv. If you
# installed the deps globally or in a differently-named env, this is fine to ignore.
if "venv" not in sys.executable.lower() and "conda" not in sys.executable.lower():
    print("\n[note] You don't appear to be in a virtual environment. That's OK, but a\n"
          "dedicated venv (see the setup cell above) avoids torch version conflicts.")

In [ ]:
# Install/verify dependencies. Skips the install entirely if everything imports (fast
# no-op on re-runs). torchcodec decodes OctoSense's .mp4 sensor video; it needs torch 2.12.
import subprocess, importlib.util as _ilu

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

_need = [m for m in ("h5py", "hdf5plugin", "cv2", "pyproj", "torchcodec", "umap") if not _ilu.find_spec(m)]
if _need:
    print("installing missing reader deps:", _need)
    _pip("huggingface_hub", "h5py", "hdf5plugin", "opencv-python-headless",
         "numpy", "pyproj", "umap-learn")
    _pip("torch==2.12.0", "torchvision==0.27.0", "torchcodec==0.14.0",
         "--index-url", "https://download.pytorch.org/whl/cpu")
    print("installed — if torch was just added/replaced, restart the kernel and re-run from the top")
else:
    print("all reader deps present — skipping install")

for pkg in ("fiftyone", "fiftyone.brain"):
    assert _ilu.find_spec(pkg), f"{pkg} missing — `pip install fiftyone==1.18.0` (see setup cell)"
print("deps ok")

In [ ]:
import os, json, math, hashlib
from pathlib import Path

import numpy as np
import h5py, hdf5plugin        # hdf5plugin registers the Blosc codec OctoSense uses
import cv2
import pyproj
from huggingface_hub import hf_hub_download
from torchcodec.decoders import VideoDecoder

import fiftyone as fo
import fiftyone.brain as fob
from fiftyone import ViewField as F

REPO = "anthonytec2/OctoSense"
print("fiftyone", fo.__version__)

## 1 · Configuration

Pick which sequences to ingest and how densely to sample them. Defaults are tuned for a *compelling
but quick* demo on a laptop: two daytime car sequences, every 8th LiDAR scan (~1.25 Hz), which yields
a few hundred richly-labeled frames total.

- **Daytime sequences** are chosen because segmentation ground truth (`has_seg`) only exists for
  daytime. If you add a night sequence, the notebook simply omits the seg field for it.
- **`SCAN_STRIDE`** trades density for speed/disk. 8 is a good demo value; drop to 1 for a dense
  single-sequence QA pass.
- We skip `events.h5` entirely (it's ~78% of the bytes and has no native FiftyOne label type).
- **`FORCE_REBUILD`** — set `True` to delete the dataset + cached frames and start completely fresh.

In [ ]:
# --- choose sequences (｢session/bag_id｣, relative to platform 'car') ------------
# These are daytime car sequences (has_seg=True) from the dataset's metadata.
SEQUENCES = [
    "car/sess7/rosbag2_2026_01_04-13_51_24",
    "car/sess8/rosbag2_2026_01_08-15_56_44",
]

SCAN_STRIDE   = 8          # take every Nth LiDAR scan as a sample
MAX_PER_SEQ   = 200        # hard cap on samples per sequence (keeps the demo snappy)
DATASET_NAME  = "octosense-demo"
WORK_DIR      = Path("./octosense_fo").resolve()   # cached frames + masks live here
UTM_EPSG      = 32618      # OctoSense NY-area data is UTM zone 18N; used for GPS -> lon/lat
FORCE_REBUILD = False      # True -> wipe dataset + cached frames and rebuild from scratch

WORK_DIR.mkdir(parents=True, exist_ok=True)

if FORCE_REBUILD:
    import shutil
    if fo.dataset_exists(DATASET_NAME):
        fo.delete_dataset(DATASET_NAME)
    if WORK_DIR.exists():
        shutil.rmtree(WORK_DIR)
    WORK_DIR.mkdir(parents=True, exist_ok=True)
    print("FORCE_REBUILD: cleared dataset and cache")

print("workspace:", WORK_DIR)

## 2 · Sequence metadata

`metadata.jsonl` (one record per sequence) is tiny and gives us the per-sequence context we'll
attach to every frame: day/night, sensor dropout, driving distance, idle fraction, and the GPS
bounding box. We pull it once (cached by `huggingface_hub` after the first fetch).

In [ ]:
mj = hf_hub_download(REPO, "metadata.jsonl", repo_type="dataset")
RECS = {r["bag_id"]: r for r in (json.loads(l) for l in open(mj))}
print(f"{len(RECS)} sequences in metadata")

for s in SEQUENCES:
    bag = s.split("/")[-1]
    r = RECS[bag]
    print(f"\n{bag}")
    print(f"  day={r['is_daytime']}  seg={r['has_seg']}  dropout={r['sensor_dropout']}"
          f"  dist={r['distance_m']:.0f}m  speed={r['mean_speed_mph']:.1f}mph"
          f"  idle={r['idle_fraction']:.2f}")

## 3 · Download the needed files per sequence

For each sequence we grab only what the demo uses: `data.h5` (sensors + calibration + GPS),
`img_left.mp4` (pixels), the depth GT, the caption file, and — for daytime — the semantic GT.
We deliberately **omit** `events.h5`, `img_right.mp4`, and `img_infrared.mp4`.

`hf_hub_download` already caches to `~/.cache/huggingface`, so a repeat call returns instantly
without re-downloading. We try a `local_files_only` lookup first so an already-cached file makes no
network call at all — this cell becomes a fast no-op once everything is local.

In [ ]:
def fetch_sequence(seq):
    # Download the subset of files this demo needs; return {name: local_path}.
    # hf_hub_download is content-addressed and cached, so existing files aren't re-fetched.
    bag = seq.split("/")[-1]
    rec = RECS[bag]
    files = ["data.h5", "img_left.mp4", "rgb_left_rect_depth.h5", "captions.h5"]
    if rec.get("has_seg"):
        files.append("rgb_left_rect_semantic.h5")
    paths = {}
    for f in files:
        # local_files_only first: instant hit if already cached, no network call
        try:
            p = hf_hub_download(REPO, f"{seq}/{f}", repo_type="dataset", local_files_only=True)
            print(f"  cached      {bag}/{f}")
        except Exception:
            print(f"  downloading {bag}/{f} ...", flush=True)
            p = hf_hub_download(REPO, f"{seq}/{f}", repo_type="dataset")
        paths[f] = p
    return paths

SEQ_PATHS = {}
for seq in SEQUENCES:
    SEQ_PATHS[seq] = fetch_sequence(seq)
print("\nall files present")

## 4 · The ingest core

This is where the dataset's structure does us a favor. Depth, segmentation, and flow are all
rendered in the **rectified left-RGB image**, and `rgb_left_rect_depth.h5` provides
`left_img_indices[k]` — the exact `img_left.mp4` frame for scan `k`. So one *sample* = one rectified
left frame at a sampled scan, and every label already registers to it.

Two things we compute at ingest:

1. **Rectification.** Raw `img_left.mp4` frames are *un-rectified*; the GT is rectified. We build the
   `stereoRectify` map once per sequence (from the calibration in `data.h5`) and remap each frame.
2. **Ego-motion flow.** Flow isn't stored — it's derived. We back-project the depth at scan `k`,
   transform by the relative camera pose to `k+1` (from `poses`), reproject, and store the per-pixel
   displacement magnitude as a heatmap. This is exact for the static scene (moving objects deviate —
   itself a useful QA signal). The implementation is numerically hardened: it filters non-finite
   depth up front, only divides by a projected depth guaranteed positive, and drops any residual
   inf/NaN before writing — so no runtime warnings and no blown-out heatmaps.

Helper functions first.

In [ ]:
# Cityscapes-19 palette (train-id -> RGB), matching the OctoSense quickstart.
CS_PALETTE = np.array([
    (128,64,128),(244,35,232),(70,70,70),(102,102,156),(190,153,153),
    (153,153,153),(250,170,30),(220,220,0),(107,142,35),(152,251,152),
    (70,130,180),(220,20,60),(255,0,0),(0,0,142),(0,0,70),
    (0,60,100),(0,80,100),(0,0,230),(119,11,32)], np.uint8)
CS_CLASSES = ["road","sidewalk","building","wall","fence","pole","traffic light",
              "traffic sign","vegetation","terrain","sky","person","rider","car",
              "truck","bus","train","motorcycle","bicycle"]
# FiftyOne Segmentation mask_targets: integer id -> class name (0..18); 255 = void
MASK_TARGETS = {i: c for i, c in enumerate(CS_CLASSES)}


def build_rectifier(h5):
    # Return (map_x, map_y, res) to rectify the left image, from calib in data.h5
    Dl = h5["img/left/dist_coeffs"][:];  Kl = h5["img/left/intrinsics"][:]
    Dr = h5["img/right/dist_coeffs"][:]; Kr = h5["img/right/intrinsics"][:]
    res = tuple(int(v) for v in h5["img/left/resolution"][:])
    imgr_T_imgl = np.linalg.inv(h5["calib/imgl_T_imgr"][:])
    rectl_R_rawl, _, P_rect_left, _ = cv2.stereoRectify(
        Kl, Dl, Kr, Dr, res,
        R=imgr_T_imgl[:3, :3], T=imgr_T_imgl[:3, -1],
        flags=cv2.CALIB_ZERO_DISPARITY, alpha=0)[:4]
    map_x, map_y = cv2.initUndistortRectifyMap(
        Kl, Dl, rectl_R_rawl, P_rect_left, res, cv2.CV_32FC1)
    return map_x, map_y, res


def nearest_idx(ts, t):
    return int(np.argmin(np.abs(np.asarray(ts) - t)))


def gps_lonlat_at(h5, t):
    # Nearest valid GPS fix (lon, lat) to time t, or None
    if "gps/data" not in h5 or "gps/t" not in h5:
        return None
    data = h5["gps/data"][:]; gt = h5["gps/t"][:]
    valid = data[:, 6] >= 0
    if valid.sum() == 0:
        return None
    k = nearest_idx(gt[valid], t)
    lat, lon = float(data[valid][k, 0]), float(data[valid][k, 1])
    return lon, lat

In [ ]:
def flow_magnitude(depth_cm, K, R3, imgl_T_ouster, poses, k):
    # Ego-motion flow magnitude image (float32, pixels) between scan k and k+1.
    # Hardened against divide-by-zero / overflow / NaN so no RuntimeWarnings and no
    # non-finite values leak into the heatmap.
    if k + 1 >= len(poses):
        return None
    m = (depth_cm > 0) & np.isfinite(depth_cm)
    if m.sum() == 0:
        return None
    RC = np.eye(4); RC[:3, :3] = R3
    rect_T_ous = RC @ imgl_T_ouster
    wTc0 = poses[k]     @ np.linalg.inv(rect_T_ous)
    wTc1 = poses[k + 1] @ np.linalg.inv(rect_T_ous)
    T = np.linalg.inv(wTc1) @ wTc0
    R, t = T[:3, :3], T[:3, 3]

    ys, xs = np.where(m)
    Z = depth_cm[ys, xs].astype(np.float32) / 100.0
    # errstate: K / pose matmuls can trip divide/overflow/invalid FP flags on some macOS
    # BLAS backends even though the result is correct — the ok-guard and isfinite(mag)
    # filter below strip any garbage before it reaches the heatmap. Silence the flags.
    with np.errstate(divide="ignore", over="ignore", invalid="ignore"):
        P = Z * (np.linalg.inv(K) @ np.stack([xs, ys, np.ones_like(xs)], 0))
        P2 = R @ P + t[:, None]

        ok = P2[2] > 1e-3                      # keep points safely in front of the camera
        if not ok.any():
            return None
        proj = K @ P2[:, ok]
        proj = proj[:2] / proj[2]              # safe: denominator > 1e-3

        du = proj[0] - xs[ok]
        dv = proj[1] - ys[ok]
        mag = np.hypot(du, dv)

    good = np.isfinite(mag)                # drop any residual inf/NaN
    out = np.zeros(depth_cm.shape, np.float32)
    yy, xx = ys[ok][good], xs[ok][good]
    out[yy, xx] = mag[good]
    return out


def save_heatmap_png(arr, lo, hi, path, gamma=1.0):
    # Quantize a float map to an 8-bit PNG on disk for on-disk Heatmap storage.
    # Values are scaled from [lo, hi] to [0, 255]; 0 stays 0 (transparent in the App).
    # gamma < 1 brightens the low end — useful for flow, where most magnitudes are small
    # so a linear map leaves them dark. Storing heatmaps on disk (map_path) instead of
    # inline (map) also avoids bloating the sample document with large arrays, which
    # serialize + zlib-compress into MongoDB slowly and can corrupt on large writes.
    a = np.asarray(arr, np.float32)
    span = max(hi - lo, 1e-6)
    q = np.clip((a - lo) / span, 0.0, 1.0)
    if gamma != 1.0:
        q = q ** gamma
    q = (q * 255.0).astype(np.uint8)
    q[a <= 0] = 0                      # preserve invalid/zero as transparent
    cv2.imwrite(str(path), q)
    return str(path)

### The per-sequence loop (idempotent)

For each sampled scan we: rectify the RGB frame and write it to disk (FiftyOne samples reference
image files on disk), write the seg mask as a PNG, and attach depth + flow heatmaps, the caption,
GPS, and all metadata. We stash the sequence's Qwen caption embedding for the nearest caption window
so we can build the caption-similarity index later.

**Caching:** each sequence writes a small `.done.json` signature, a `manifest.json`, a `qwen_emb.npy`,
and per-frame **PNG** depth/flow maps into its cache folder once ingested. On re-run, a
sequence already ingested *with the same stride/cap* is rebuilt from those sidecars — no video
decode, no rectify, no HDF5 reads.

The depth and flow heatmaps are stored **on disk** (`fo.Heatmap(map_path=...)`) rather than inline
(`map=<array>`). Inline heatmaps serialize the full float array into the sample's MongoDB document
and zlib-compress it; for arrays this large that path is slow and can corrupt on interrupted writes.
On-disk PNGs sidestep that entirely and load faster in the App. Each PNG is a quantization of the
map to 0–255 over its display range (depth over 0–60 m, flow over 0–80th-percentile with a gamma
boost so small ego-motion still reads), so the label carries `range=[0, 255]`.

In [ ]:
def _seq_signature(seq):
    # what makes an ingested sequence "current" — bump v if you change sampling logic
    return {"seq": seq, "stride": SCAN_STRIDE, "max": MAX_PER_SEQ, "v": 5}


def ingest_sequence(seq, paths, samples_out, qwen_rows):
    bag = seq.split("/")[-1]
    rec = RECS[bag]
    seq_dir = WORK_DIR / bag
    done_path = seq_dir / ".done.json"
    emb_path  = seq_dir / "qwen_emb.npy"
    man_path  = seq_dir / "manifest.json"

    # --- fast path: already ingested with the same settings -> load from cache
    if done_path.exists() and emb_path.exists() and man_path.exists():
        try:
            sig_ok = json.loads(done_path.read_text()) == _seq_signature(seq)
        except Exception:
            sig_ok = False
        if sig_ok:
            manifest = json.loads(man_path.read_text())   # list of per-sample dicts
            embs = np.load(emb_path)
            for row, e in zip(manifest, embs):
                s = fo.Sample(filepath=row["filepath"])
                s["depth"] = fo.Heatmap(map_path=row["depth_png"], range=[0, 255])
                if row.get("flow_png"):
                    s["flow_mag"] = fo.Heatmap(map_path=row["flow_png"], range=[0, 255])
                if row.get("seg_path"):
                    s["segmentation"] = fo.Segmentation(mask_path=row["seg_path"])
                for kf, vf in row["fields"].items():
                    s[kf] = vf
                if row.get("location"):
                    s["location"] = fo.GeoLocation(point=row["location"])
                samples_out.append(s)
                qwen_rows.append((row["filepath"], e.astype(np.float32)))
            print(f"  {bag}: loaded {len(manifest)} samples from cache (skipped ingest)")
            return

    # --- slow path: (re)ingest
    (seq_dir / "rgb").mkdir(parents=True, exist_ok=True)
    (seq_dir / "seg").mkdir(parents=True, exist_ok=True)
    (seq_dir / "arr").mkdir(parents=True, exist_ok=True)

    h5   = h5py.File(paths["data.h5"], "r")
    dep  = h5py.File(paths["rgb_left_rect_depth.h5"], "r")
    has_seg = "rgb_left_rect_semantic.h5" in paths
    seg  = h5py.File(paths["rgb_left_rect_semantic.h5"], "r") if has_seg else None
    caps = h5py.File(paths["captions.h5"], "r")

    cap_text = caps["captions"][:]
    cap_meta = caps["metadata"][:]
    cap_emb  = caps["embeddings"][:]           # (W, 4096) Qwen3
    cap_frame = cap_meta["frame_idx"].astype(int)

    vid = VideoDecoder(paths["img_left.mp4"])
    map_x, map_y, res = build_rectifier(h5)

    K  = dep["K_rect"][:]
    R3 = dep["R3"][:]
    left_idx = dep["left_img_indices"][:]
    poses    = dep["poses"][:]
    imgl_T_ouster = h5["ouster/imgl_T_ouster"][:]
    ouster_t = h5["ouster/t"][:]

    n_scans = len(left_idx)
    picks = list(range(0, n_scans - 1, SCAN_STRIDE))[:MAX_PER_SEQ]
    print(f"  {bag}: {n_scans} scans -> {len(picks)} samples (seg={has_seg})")

    manifest, emb_list = [], []
    for k in picks:
        fidx = int(left_idx[k])
        raw = vid[fidx].permute(1, 2, 0).cpu().numpy()          # HWC uint8
        rectl = cv2.remap(raw, map_x, map_y, cv2.INTER_LINEAR)
        rgb_path = seq_dir / "rgb" / f"{k:06d}.jpg"
        cv2.imwrite(str(rgb_path), cv2.cvtColor(rectl, cv2.COLOR_RGB2BGR),
                    [cv2.IMWRITE_JPEG_QUALITY, 92])

        s = fo.Sample(filepath=str(rgb_path))

        # depth heatmap (meters) -> on-disk PNG (map_path), range mapped to 0..255.
        # Real depth ceiling stays 60 m; the label range is [0,255] because the PNG is
        # a quantized view of [0, 60] m.
        depth_cm = dep["depth_cm"][k].astype(np.float32)
        depth_m = np.where(depth_cm > 0, depth_cm / 100.0, 0.0)
        depth_png = seq_dir / "arr" / f"{k:06d}_depth.png"
        save_heatmap_png(depth_m, 0.0, 60.0, depth_png)
        s["depth"] = fo.Heatmap(map_path=str(depth_png), range=[0, 255])

        # ego-motion flow magnitude heatmap -> on-disk PNG
        flow_png, flow_hi = None, None
        fm = flow_magnitude(depth_cm, K, R3, imgl_T_ouster, poses, k)
        if fm is not None and fm.max() > 0:
            # 80th pct ceiling (not 95th): most flow is small ego-motion, so a lower
            # ceiling spreads the interesting mid-range across the full colormap instead
            # of compressing it into the dark low end.
            flow_hi = float(np.percentile(fm[fm > 0], 80)) or 1.0
            flow_png = seq_dir / "arr" / f"{k:06d}_flow.png"
            save_heatmap_png(fm, 0.0, flow_hi, flow_png, gamma=0.5)
            s["flow_mag"] = fo.Heatmap(map_path=str(flow_png), range=[0, 255])

        # semantic segmentation mask -> PNG on disk
        seg_path = None
        if has_seg:
            sem = seg["semantic"][k].astype(np.uint8)          # train-ids, >=19/255 void
            sem_store = sem.copy(); sem_store[sem_store >= 19] = 255
            seg_path = seq_dir / "seg" / f"{k:06d}.png"
            cv2.imwrite(str(seg_path), sem_store)
            s["segmentation"] = fo.Segmentation(mask_path=str(seg_path))

        # caption (nearest caption window by frame index)
        wi = int(np.argmin(np.abs(cap_frame - fidx)))
        caption = cap_text[wi].decode() if isinstance(cap_text[wi], bytes) else str(cap_text[wi])
        s["caption"] = caption

        # GPS -> map point
        t = float(ouster_t[k])
        ll = gps_lonlat_at(h5, t)
        if ll is not None:
            s["location"] = fo.GeoLocation(point=[ll[0], ll[1]])

        fields = {
            "t": t, "scan_idx": k, "frame_idx": fidx,
            "speed_mps": float(cap_meta["speed_mps"][wi]),
            "turn_deg":  float(cap_meta["turn_deg"][wi]),
            "is_night":  bool(cap_meta["is_night"][wi]),
            "caption":   caption,
            "sequence":  bag, "session": rec["session"],
            "is_daytime": bool(rec["is_daytime"]),
            "sensor_dropout": rec["sensor_dropout"],
            "seq_distance_m": float(rec["distance_m"]),
            "seq_idle_frac":  float(rec["idle_fraction"]),
        }
        for kf, vf in fields.items():
            s[kf] = vf

        samples_out.append(s)
        qwen_rows.append((str(rgb_path), cap_emb[wi].astype(np.float32)))

        manifest.append({
            "filepath": str(rgb_path),
            "depth_png": str(depth_png),
            "flow_png": str(flow_png) if flow_png else None,
            "seg_path": str(seg_path) if seg_path else None,
            "location": [ll[0], ll[1]] if ll is not None else None,
            "fields": fields,
        })
        emb_list.append(cap_emb[wi].astype(np.float32))

    for fh in (h5, dep, caps):
        fh.close()
    if seg is not None:
        seg.close()

    # persist cache sidecars
    np.save(emb_path, np.stack(emb_list))
    man_path.write_text(json.dumps(manifest))
    done_path.write_text(json.dumps(_seq_signature(seq)))

## 5 · Build the FiftyOne dataset (reuse if already built)

If a dataset with this name already exists *and* holds the sequences we expect, we reuse it as-is.
Otherwise we build it from the (possibly cached) ingest above. Either way we reconstruct `qwen_rows`
cheaply from the per-sequence caches so Section 7 can build the caption index.

In [ ]:
def dataset_is_current(ds):
    if ds is None:
        return False
    try:
        have = set(ds.distinct("sequence"))
    except Exception:
        return False
    want = {s.split("/")[-1] for s in SEQUENCES}
    return want.issubset(have)


dataset = fo.load_dataset(DATASET_NAME) if fo.dataset_exists(DATASET_NAME) else None
samples, qwen_rows = [], []

if dataset_is_current(dataset) and not FORCE_REBUILD:
    print(f"reusing existing dataset '{DATASET_NAME}' ({len(dataset)} samples)")
    # rebuild qwen_rows from the per-sequence caches (no frame work)
    for seq in SEQUENCES:
        bag = seq.split("/")[-1]
        emb_path = WORK_DIR / bag / "qwen_emb.npy"
        man_path = WORK_DIR / bag / "manifest.json"
        if emb_path.exists() and man_path.exists():
            manifest = json.loads(man_path.read_text())
            embs = np.load(emb_path)
            for row, e in zip(manifest, embs):
                qwen_rows.append((row["filepath"], e.astype(np.float32)))
else:
    if fo.dataset_exists(DATASET_NAME):
        fo.delete_dataset(DATASET_NAME)
    dataset = fo.Dataset(DATASET_NAME, persistent=True)
    for seq in SEQUENCES:
        ingest_sequence(seq, SEQ_PATHS[seq], samples, qwen_rows)
    dataset.add_samples(samples)
    dataset.mask_targets = {"segmentation": MASK_TARGETS}
    dataset.default_mask_targets = MASK_TARGETS
    dataset.save()
    print(f"\nbuilt dataset '{DATASET_NAME}' with {len(dataset)} samples")

# High-contrast per-field value colormaps. Two requirements for these to actually apply:
#   1. color_by="value"  — in the default "field" mode the App draws each heatmap as one
#      flat color and never consults a colormap.
#   2. colorscales=[...]  — heatmap colormaps live in this top-level list keyed by field
#      path (NOT fields[].colorscale). flow_mag on "viridis" gives the blue→green→yellow
#      ramp that reads ego-motion magnitude cleanly; depth on "inferno" stays distinct.
# NOTE: some App builds don't honor the dataset-level scheme, so we ALSO set it on the
# session at launch (Section 9) — belt and suspenders.
dataset.app_config.color_scheme = fo.ColorScheme(
    color_by="value",
    colorscales=[
        {"path": "flow_mag", "name": "viridis"},   # blue→green→yellow: reads the
                                                    # ego-motion gradient beautifully
        {"path": "depth",    "name": "inferno"},    # distinct from flow so overlays
                                                    # don't collide
    ],
)
dataset.save()

print(dataset)

## 6 · CLIP similarity index — App-native text search + embedding plot

A CLIP index over the rectified frames gives two things at once: natural-language prompt search
inside the App (the Similarity Search panel and `sort_by_similarity`), and the embeddings that feed
a 2D UMAP visualization you can lasso to select clusters.

Both brain runs are **skipped if their `brain_key` already exists** on the dataset, so re-running is
instant. Delete a key (`dataset.delete_brain_run(key)`) to force a recompute.

In [ ]:
existing = set(dataset.list_brain_runs())

if "clip_sim" in existing:
    clip_index = dataset.load_brain_results("clip_sim")
    print("reusing CLIP similarity index 'clip_sim'")
else:
    clip_index = fob.compute_similarity(
        dataset,
        model="clip-vit-base32-torch",
        embeddings="clip_emb",          # cache embeddings on the samples for reuse
        brain_key="clip_sim",
    )
    print("built CLIP similarity index 'clip_sim'")

print("supports text prompts:", clip_index.config.supports_prompts)

In [ ]:
# 2D embedding visualization (UMAP) — appears as a Panel in the App.
# If umap-learn is troublesome on your Mac, switch method to "tsne".
if "clip_viz" in set(dataset.list_brain_runs()):
    print("reusing embedding visualization 'clip_viz'")
else:
    fob.compute_visualization(
        dataset,
        embeddings="clip_emb",
        method="umap",
        brain_key="clip_viz",
    )
    print("built embedding visualization 'clip_viz'")

## 7 · Qwen caption index — search the dataset's *own* embeddings

OctoSense captions each window with a Gemma VLM and embeds that caption with Qwen3-Embedding-8B
(4096-d). Those are text-space vectors, so we register them as a **precomputed** similarity index
(no model, `sklearn` backend). This lets you pick a frame and retrieve others whose *scene
description* is nearest — a semantically different query than CLIP's visual similarity.

Skipped if `caption_sim` already exists. Two robustness details: if the dataset was built by an
older run without cache sidecars, the vectors are rebuilt directly from `captions.h5` via each
sample's `frame_idx`; and the sklearn cosine build is wrapped in `np.errstate(...)` to silence
spurious float32-matmul warnings some macOS BLAS backends emit (the inputs are verified finite and
non-zero, so the results are correct — see the inline comment).

In [ ]:
if "caption_sim" in set(dataset.list_brain_runs()):
    caption_index = dataset.load_brain_results("caption_sim")
    print("reusing caption similarity index 'caption_sim' "
          f"(size {caption_index.total_index_size})")
else:
    fp_to_id = {s.filepath: s.id for s in dataset.select_fields("filepath")}

    # Preferred: use the qwen_rows assembled in Section 5 (from the ingest/cache).
    ids, vecs = [], []
    for fp, v in qwen_rows:
        if fp in fp_to_id:
            ids.append(fp_to_id[fp]); vecs.append(v)

    # Fallback: dataset was built by an older run with no cache sidecars, so qwen_rows
    # is empty. Rebuild the Qwen vectors straight from captions.h5 using each sample's
    # stored frame_idx to pick the nearest caption window. Works regardless of how the
    # dataset was originally built.
    if not vecs:
        print("qwen_rows empty — rebuilding caption embeddings from captions.h5")
        for seq in SEQUENCES:
            bag = seq.split("/")[-1]
            caps = h5py.File(SEQ_PATHS[seq]["captions.h5"], "r")
            cap_emb   = caps["embeddings"][:]
            cap_frame = caps["metadata"]["frame_idx"].astype(int)
            caps.close()
            view = dataset.match(F("sequence") == bag).select_fields(["filepath", "frame_idx"])
            for s in view:
                wi = int(np.argmin(np.abs(cap_frame - int(s.frame_idx))))
                ids.append(s.id); vecs.append(cap_emb[wi].astype(np.float32))

    assert vecs, "no Qwen vectors — check SEQ_PATHS / captions.h5 availability"
    qwen_emb = np.stack(vecs).astype(np.float32)
    print("qwen embeddings:", qwen_emb.shape)

    # Note on the errstate wrapper: the input vectors are verified finite and non-zero
    # (checked directly: 0 zero-norm, 0 non-finite rows). The divide/overflow/invalid
    # RuntimeWarnings that sklearn can emit here come from spurious floating-point flags
    # tripped inside its float32 cosine matmul on some macOS BLAS backends — not from bad
    # data, and the results are numerically correct. We suppress those flags at this
    # known-safe call site rather than mutate correct inputs.
    with np.errstate(divide="ignore", over="ignore", invalid="ignore"):
        caption_index = fob.compute_similarity(
            dataset,
            embeddings=qwen_emb,      # precomputed vectors -> no model needed
            sample_ids=ids,
            metric="cosine",
            backend="sklearn",
            brain_key="caption_sim",
        )
    print("built caption similarity index 'caption_sim' "
          f"(size {caption_index.total_index_size})")

## 8 · Saved views + search, ready to demo

This builds a set of **saved views** and saves them onto the dataset, so at demo time they show up
in the App's view dropdown (top-left) and you can jump between them with one click — no typing filters
live. Each is also a plain FiftyOne view you can assign to `session.view` from code.

It also runs a quick **natural-language search** check. At demo time you'll do text search right in
the App's **Text Search** box (Section 9) — type a prompt against `clip_sim` and the grid reorders.
This cell just verifies the same query works from code too, which is handy for scripting or as a
fallback. Image-to-image similarity (the App's **Image Search** button) works directly in the App
for both indexes.

In [ ]:
# Build the demo views and SAVE them onto the dataset so they appear in the App's
# view dropdown (top-left) for one-click switching during a presentation.

demo_views = {
    # Moving frames with segmentation present — the meat of a seg-QA pass (hard scenes first)
    "QA: seg + moving (hard turns first)": (
        dataset
        .match(F("segmentation") != None)
        .match(F("speed_mps") > 2.0)
        .sort_by("turn_deg", reverse=True)
    ),
    # Fast-moving frames — THE frames to demo flow on (flow scales with ego-motion, so a
    # near-stopped frame shows almost none). View flow_mag alone on one of these.
    "Flow demo: fast-moving": (
        dataset.match(F("speed_mps") > 5.0).sort_by("speed_mps", reverse=True)
    ),
    # Near-stationary frames (idle / stop-and-go) — often redundant, prune candidates
    "Prune: near-stationary": dataset.match(F("speed_mps") < 0.5),
    # Sharpest turns in the dataset — cornering scenes
    "Sharp turns": dataset.sort_by(abs(F("turn_deg")), reverse=True),
}

for name, view in demo_views.items():
    if name in dataset.list_saved_views():
        dataset.delete_saved_view(name)
    dataset.save_view(name, view)
    print(f"saved view  {name!r:42s} ({len(view)} samples)")

print("\nsaved views:", dataset.list_saved_views())

In [ ]:
# Quick check that natural-language search works (you'll do this live in the App's Text
# Search box; this just confirms the same query resolves from code). errstate silences
# spurious float32-matmul warnings on some macOS BLAS backends (results are correct).
with np.errstate(divide="ignore", over="ignore", invalid="ignore"):
    hits = dataset.sort_by_similarity("cars turning left", k=25, brain_key="clip_sim")
    rows = [(s.sequence, s.scan_idx) for s in hits.select_fields(["sequence", "scan_idx"])[:12]]
print("top CLIP matches for 'cars turning left':", rows)

## 9 · Launch the App

This opens the FiftyOne App. Things to try:

- **Filter** in the left sidebar by `is_night`, `sensor_dropout`, `speed_mps`, `turn_deg`,
  `sequence` — then watch the grid and the map update together.
- **Label QA:** open a sample and toggle the `segmentation` mask and `depth` / `flow_mag` heatmaps
  **one at a time** (stacked, they blend into a wash and the individual signals disappear). The
  fields carry distinct value-colormaps — `flow_mag` on `viridis` (blue→green→yellow),
  `depth` on `inferno` — set with **Color by: value** so the colormaps actually render (in the
  default "field" mode heatmaps draw as one flat color). The scheme is applied both on the dataset
  and on the session at launch, so it comes up correctly without touching the color-settings panel.
  **Demo flow on a fast-moving frame** (use the "Flow demo: fast-moving" saved view) — flow
  magnitude scales with ego-motion, so on a near-stopped frame it's near-zero and looks dim no matter
  the colormap. On a moving frame, `flow_mag` alone shows a gorgeous radial gradient on `viridis`:
  bright yellow-green on the fast-moving foreground road, cool blue toward the distant focus of
  expansion — you can read the car's forward motion right off the colors. A genuinely moving vehicle
  **breaks** that gradient, which is the **flow gotcha** / QA signal. Also look for seg bleeding at
  object edges and depth holes / LiDAR blooming around retroreflective signs.
- **Saved views:** the view dropdown (top-left) has the demo views from Section 8 — "QA: seg +
  moving", "Flow demo: fast-moving", "Prune: near-stationary", "Sharp turns". One click to jump to
  each; great for a scripted walkthrough.
- **Embeddings panel:** open the `clip_viz` plot, lasso a cluster, see those frames select in the
  grid. Night vs day and highway vs residential usually separate cleanly.
- **Search — two brains, two ways:**
  - *Visual similarity (Image Search):* select a frame, hit **Image Search** with `clip_sim` for
    visual look-alikes; run it again from the same frame with `caption_sim` for scene-*description*
    matches. Same seed, two different rankings — that's the two-brains story, and it works directly
    in the App.
  - *Natural-language (text) search:* open the **Similarity Search** panel, pick `clip_sim`, and
    type a query in the **Text Search** box — "car turning left", "pedestrian crossing", "snow on the
    road". The grid reorders to the best matches. (If you ever run in an environment where the box
    errors, the `session.view = ...` cell below does the same thing from code as a backup.)
- **Map panel:** the GPS points render on a basemap; pan around to see where the drives happened.

In [ ]:
session = fo.launch_app(dataset)

# Belt-and-suspenders: some App builds don't pick up the dataset-level color scheme, so
# set it on the session too. This is what guarantees flow_mag comes up value-colored on
# "viridis" without anyone touching the color-settings panel by hand.
session.color_scheme = fo.ColorScheme(
    color_by="value",
    colorscales=[
        {"path": "flow_mag", "name": "viridis"},
        {"path": "depth",    "name": "inferno"},
    ],
)
session

### Optional: drive text search from code

Natural-language search is easiest right in the App's Text Search box (above). This cell does the
same thing from code — handy for scripting a demo, or as a fallback in any environment where the
App's box misbehaves. Re-run with different query strings; the grid reorders to the matches.

In [ ]:
QUERY = "cars turning left"   # try: "pedestrian crossing", "snow on the road", "parked cars"
with np.errstate(divide="ignore", over="ignore", invalid="ignore"):
    session.view = dataset.sort_by_similarity(QUERY, k=25, brain_key="clip_sim")
print(f"showing top 25 matches for {QUERY!r} — see the App grid")

### Jump to a saved view (run live)

Or switch the App to any of the saved demo views from code (they're also in the App's view
dropdown).

In [ ]:
# session.view = dataset.load_saved_view("Flow demo: fast-moving")
# session.view = dataset.load_saved_view("QA: seg + moving (hard turns first)")
# session.clear_view()   # back to the full dataset
print("saved views available:", dataset.list_saved_views())

---
### Re-running & cleanup

**Re-running:** just execute top to bottom. Downloads, ingested frames, and all three brain runs are
cached and skipped automatically — only genuinely new work runs. To force specific recomputes:

```python
# rebuild everything from scratch:  set FORCE_REBUILD = True in Section 1

# recompute a single brain run:
# dataset.delete_brain_run("clip_sim")     # or "clip_viz" / "caption_sim"

# re-ingest one sequence (e.g. after changing SCAN_STRIDE):
#   FORCE_REBUILD=True is simplest; or delete its cache folder:
# import shutil; shutil.rmtree(WORK_DIR / "rosbag2_2026_01_04-13_51_24")
```

**Full cleanup:**

```python
# fo.delete_dataset(DATASET_NAME)
# import shutil; shutil.rmtree(WORK_DIR)   # removes cached frames/masks/arrays
```

**Scaling up.** To ingest more of OctoSense: add bag ids to `SEQUENCES` (mix in night sequences —
the notebook auto-skips seg for them), lower `SCAN_STRIDE` for denser coverage, and raise
`MAX_PER_SEQ`. For hundreds of sequences, swap the `sklearn` caption backend for a real vector DB
(Qdrant/Mongo) via the `backend=` argument to `compute_similarity`, and consider a delegated /
batched ingest rather than the in-notebook loop.
